## 🎯 Learning Objectives
* Understand the core principles and applications of image-to-image (img2img) generation in Stable Diffusion.
* Learn how to effectively utilize inpainting techniques for targeted and seamless image modification.
* Implement practical img2img and inpainting workflows using modern Stable Diffusion libraries and models.
* Evaluate the performance characteristics, trade-offs, and production-ready use cases for these advanced image generation methods.


## Image-to-Image and Inpainting Workflows: Precision Control for Image Generation

In the realm of generative AI, `text-to-image` models have revolutionized content creation. However, often we don't want to generate an image from scratch; we want to modify, enhance, or transform an *existing* image. This is where **Image-to-Image (img2img)** and **Inpainting** workflows become indispensable, offering unparalleled control and precision.

### 1. Image-to-Image (Img2Img): Guiding the Creative Process

Imagine you're a sculptor. With `text-to-image`, you start with a block of raw clay and shape it entirely based on your vision. With `img2img`, you're given a partially formed sculpture – perhaps a rough outline or a base shape – and your task is to refine it, add details, or even transform its style based on a new artistic direction. The existing image acts as a powerful initial constraint, guiding the diffusion process rather than letting it wander freely from pure noise.

**How it works:**
1.  **Input Image:** You provide an existing image as a starting point.
2.  **Noise Injection:** The input image is first 'noised' to a certain degree. This process essentially adds controlled randomness, allowing the model to deviate from the original. The `denoising_strength` (or `strength`) parameter dictates how much noise is added: a low strength means subtle changes, while a high strength allows for significant transformations.
3.  **Diffusion Process:** This partially noised image then serves as the initial latent representation for the Stable Diffusion model. Guided by your text prompt, the model iteratively denoises this latent, gradually transforming it into a new image that aligns with both the prompt and the structure/content of the original image (to a degree determined by `denoising_strength`).

**Real-world Applications (2026 Context):**
*   **Style Transfer:** Turning a photograph into a painting in the style of Van Gogh or a comic book illustration.
*   **Image Variations:** Generating multiple creative variations of a product shot or a character design while retaining core elements.
*   **Sketch-to-Image:** Transforming rough sketches or wireframes into photorealistic renders or detailed concept art.
*   **Upscaling and Enhancement:** Improving the resolution and detail of lower-quality images, adding stylistic flair.

### 2. Inpainting: Surgical Precision for Image Editing

Inpainting is a specialized form of `img2img` that offers even finer control. Think of it as digital surgery. Instead of modifying the entire image, you specify a particular region (using a mask) that needs to be changed, while leaving the rest of the image untouched. The model then intelligently 'fills in' the masked area, ensuring it blends seamlessly with the surrounding content and adheres to your instructions.

**How it works:**
1.  **Input Image & Mask:** You provide an image and a corresponding binary mask. The mask indicates which pixels should be modified (e.g., white for masked, black for unmasked).
2.  **Masked Noise:** The region covered by the mask is heavily noised, effectively erasing its original content. The unmasked regions are preserved.
3.  **Contextual Generation:** The Stable Diffusion model then performs a diffusion process, but critically, it's constrained by the unmasked regions. It uses the surrounding pixels as context to generate new content within the masked area that is consistent with the prompt and the visual flow of the image.

**Real-world Applications (2026 Context):**
*   **Object Removal:** Seamlessly removing unwanted elements like photobombers, power lines, or watermarks (with ethical considerations).
*   **Object Addition/Replacement:** Adding new elements (e.g., a hat, glasses, a different background object) or replacing existing ones (e.g., changing a car's color, swapping a shirt).
*   **Image Restoration:** Repairing damaged or incomplete photographs by filling in missing sections.
*   **Virtual Try-On:** Modifying clothing on a model in an e-commerce setting.
*   **Scene Editing:** Changing weather conditions in a specific part of a landscape, or altering architectural details.

Both `img2img` and `inpainting` leverage the power of Stable Diffusion to move beyond mere generation, enabling sophisticated and controlled image manipulation. These techniques are fundamental for building advanced, production-ready image pipelines.


In [ ]:
import torch
from PIL import Image, ImageDraw
from diffusers import StableDiffusionXLImg2ImgPipeline, StableDiffusionXLInpaintPipeline
import requests
from io import BytesIO

# --- Configuration --- #
# Using a modern, efficient model. SDXL is a strong candidate for 2026 production pipelines.
# Ensure you have sufficient VRAM (e.g., 12GB+ for SDXL in bfloat16).
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Helper Functions --- #
def load_image_from_url(url):
    """Loads an image from a URL."""
    response = requests.get(url)
    return Image.open(BytesIO(response.content)).convert("RGB")

def display_images(images, titles=None):
    """Displays a list of PIL images in a grid."""
    import matplotlib.pyplot as plt
    num_images = len(images)
    fig, axes = plt.subplots(1, num_images, figsize=(5 * num_images, 5))
    if num_images == 1:
        axes = [axes] # Ensure axes is iterable for single image case
    for i, img in enumerate(images):
        axes[i].imshow(img)
        axes[i].axis('off')
        if titles and len(titles) > i:
            axes[i].set_title(titles[i])
    plt.tight_layout()
    plt.show()

print(f"Using device: {DEVICE}")
print(f"Loading model: {MODEL_ID}")

# --- 1. Image-to-Image Workflow --- #
print("\n--- Running Image-to-Image (Img2Img) Example ---")

# Load the img2img pipeline
# Using bfloat16 for memory efficiency, crucial for large models like SDXL on modern GPUs.
img2img_pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    variant="fp16", # Use fp16 weights for faster inference if available
    use_safetensors=True
).to(DEVICE)

# Example input image (a cityscape)
img_url = "https://huggingface.co/datasets/diffusers/docs-images/resolve/main/img2img_7.png"
init_image = load_image_from_url(img_url).resize((768, 512))

# Prompt to guide the transformation
prompt_img2img = "A futuristic city, cyberpunk style, neon lights, rain, highly detailed, cinematic lighting"

# Denoising strength: 0.0 means no change, 1.0 means complete re-generation (like text2img)
# A value around 0.6-0.8 is often good for significant style transfer while retaining structure.
denoising_strength = 0.75

print(f"Generating img2img with prompt: '{prompt_img2img}' and denoising_strength: {denoising_strength}")

# Generate the image
# Using a higher guidance_scale for stronger adherence to the prompt.
# num_inference_steps can be adjusted for quality vs. speed trade-off.
img2img_output = img2img_pipe(
    prompt=prompt_img2img,
    image=init_image,
    strength=denoising_strength,
    guidance_scale=7.5,
    num_inference_steps=30
).images[0]

display_images([init_image, img2img_output], titles=["Original Image", "Img2Img Output"])

# Clean up img2img pipeline to free VRAM
del img2img_pipe
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# --- 2. Inpainting Workflow --- #
print("\n--- Running Inpainting Example ---")

# Load the inpainting pipeline
inpaint_pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    variant="fp16",
    use_safetensors=True
).to(DEVICE)

# Example input image (a person in a park)
inpaint_img_url = "https://huggingface.co/datasets/diffusers/docs-images/resolve/main/sd_xl_inpaint_example.png"
init_inpaint_image = load_image_from_url(inpaint_img_url).resize((768, 768))

# Create a mask image
# We'll mask out the person's face to replace it with a robot face.
mask_image = Image.new("RGB", init_inpaint_image.size, (0, 0, 0)) # Black background
draw = ImageDraw.Draw(mask_image)

# Define the bounding box for the mask (approximate face region)
x1, y1 = 300, 150
x2, y2 = 450, 300
draw.rectangle([x1, y1, x2, y2], fill=(255, 255, 255)) # White rectangle for the mask

# Prompt for inpainting
prompt_inpaint = "A robot face, highly detailed, metallic, glowing eyes"

print(f"Generating inpainting with prompt: '{prompt_inpaint}'")

# Generate the inpainted image
# For inpainting, strength is often implicitly handled or less critical than for img2img.
# The mask itself defines the area of change.
inpaint_output = inpaint_pipe(
    prompt=prompt_inpaint,
    image=init_inpaint_image,
    mask_image=mask_image,
    guidance_scale=7.5,
    num_inference_steps=30
).images[0]

display_images(
    [init_inpaint_image, mask_image.convert("L"), inpaint_output],
    titles=["Original Image", "Mask", "Inpaint Output"]
)

# Clean up inpaint pipeline
del inpaint_pipe
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("\n--- Examples Complete ---")


### Interpreting the Output and Practical Considerations

#### Interpreting Code Output

*   **Img2Img Output:** Observe how the `Img2Img Output` image retains the general composition and elements of the `Original Image` but has been significantly transformed in style and detail according to the `prompt_img2img`. The `denoising_strength` parameter is key here: a lower value would result in an output closer to the original, while a higher value would allow for more drastic changes, potentially losing some original structure.
*   **Inpaint Output:** Notice how only the masked region (the face in our example) has been altered, while the rest of the image remains largely unchanged. The new content (the robot face) should seamlessly blend with the surrounding environment, demonstrating the model's ability to understand context and generate coherent additions. The `mask_image` clearly shows the area targeted for modification.

#### Performance Trade-offs

In 2026, while models are more efficient, performance remains a critical factor for production pipelines:

*   **Computational Cost:** Both `img2img` and `inpainting` involve running the full diffusion process, similar to `text2img`. The primary factors influencing cost are the model size (e.g., SDXL is larger than SD 1.5), `num_inference_steps`, and image resolution. Higher resolution and more steps lead to longer generation times.
*   **Memory (VRAM):** Loading large models like SDXL, especially with higher resolution images, demands substantial VRAM. Using `torch.bfloat16` (or `fp16`) is crucial for reducing memory footprint and enabling larger batch sizes or higher resolutions on consumer-grade GPUs (e.g., 12GB+ for SDXL). For extreme cases, techniques like model offloading (moving parts of the model to CPU when not in use) or distributed inference are employed.
*   **Latency:** For real-time or near real-time applications (e.g., live editing tools, interactive content generation), latency is paramount. This often necessitates highly optimized pipelines, specialized hardware (e.g., NVIDIA's latest GPUs, custom NPUs, or cloud-based inference services with dedicated accelerators), and potentially smaller, fine-tuned models.
*   **Quality vs. Speed:** There's always a trade-off. More `num_inference_steps` generally yields higher quality but increases generation time. `guidance_scale` also impacts quality and adherence to the prompt, but excessively high values can lead to artifacts.

#### Typical Use Cases in Production Pipelines (2026)

*   **E-commerce & Advertising:**
    *   **Product Variations:** Generating different colors, textures, or materials for a product from a single base image.
    *   **Virtual Try-On:** Inpainting clothing onto models or users for personalized shopping experiences.
    *   **Dynamic Ad Creatives:** Automatically adapting product images to various seasonal themes or target demographics.
*   **Creative Industries (Gaming, Film, Design):**
    *   **Concept Art Iteration:** Rapidly generating variations of character designs, environments, or props from initial sketches or existing assets.
    *   **Asset Generation:** Creating diverse textures, materials, or environmental elements for 3D models.
    *   **Post-Production Editing:** Removing unwanted elements from footage, extending backgrounds, or subtly altering scene details.
*   **Automated Content Generation:**
    *   **Personalized Marketing:** Generating unique images for individual users based on their preferences or data.
    *   **News & Media:** Quickly creating illustrative images for articles or social media posts based on existing visual assets.
*   **Image Restoration & Enhancement:**
    *   **Archival Digitization:** Repairing damage, removing dust/scratches, or colorizing old photographs.
    *   **Upscaling & Detail Enhancement:** Improving the perceived resolution and adding intricate details to lower-resolution images for print or high-definition displays.

These advanced control techniques are fundamental building blocks for sophisticated AI-powered visual content creation systems, enabling developers and creators to move beyond basic generation to highly controlled and targeted image manipulation.


### Resources for Further Learning

To deepen your understanding and explore more advanced applications of Stable Diffusion, img2img, and inpainting, consult the following resources:

*   **Hugging Face `diffusers` Library Documentation:**
    *   [Main Documentation](https://huggingface.co/docs/diffusers/index)
    *   [Image-to-Image Pipeline](https://huggingface.co/docs/diffusers/api/pipelines/stable_diffusion/img2img)
    *   [Inpainting Pipeline](https://huggingface.co/docs/diffusers/api/pipelines/stable_diffusion/inpaint)
    *   [Stable Diffusion XL](https://huggingface.co/docs/diffusers/api/pipelines/stable_diffusion/stable_diffusion_xl)

*   **Stability AI:**
    *   [Official Website](https://stability.ai/)
    *   [Stable Diffusion XL Research Paper](https://stability.ai/research/stable-diffusion-xl-a-leap-in-realistic-image-generation-and-artistic-control)

*   **PyTorch Documentation:**
    *   [Official Website](https://pytorch.org/docs/stable/index.html)
    *   Essential for understanding the underlying deep learning framework.

*   **Google AI Studio / Gemini API:**
    *   [Google AI Studio](https://aistudio.google.com/)
    *   While not directly Stable Diffusion, it provides a broader context for generative AI development and API integration, which is relevant for production systems.

*   **Academic Papers & Research:**
    *   Explore papers on diffusion models, conditional image generation, and specific inpainting techniques on platforms like arXiv or Google Scholar to stay updated on the latest advancements.
